# P15 · Regímenes macroeconómicos y diversificación
**QuantLab · Módulo 1 · Lecciones 1.23 y 1.29**

## 1. Objetivos
- Construir una cartera de activos con retornos en EUR y medir su riesgo mediante covarianzas.
- Separar exposición sistemática a un benchmark de riesgo residual en un modelo lineal.
- Definir regímenes monetarios e investigar retornos, correlaciones y riesgo condicionales.
- Alinear las variables macroeconómicas con su disponibilidad real, no con el mes al que se refieren.

Modo inicial: **sintético, reproducible**. No representa el mercado ni constituye una recomendación. En modo CSV podrás utilizar las series de P14 y un histórico macroeconómico point-in-time. Ninguna celda envía órdenes.

Las funciones están en **`src/quantlab/p15/regimes.py`** y sus pruebas en **`tests/test_regimes.py`** (`pytest` desde la raíz del proyecto). Los ejercicios están resueltos en la sección 8.

## 2. Configuración
Colab, Jupyter y VS Code: requiere numpy, pandas, matplotlib e IPython. Si faltan, instala con `%pip install numpy pandas matplotlib`. Los retornos son decimales, los tipos y la inflación se expresan en puntos porcentuales, y el balance en miles de millones de USD. Las semanas terminan el viernes (etiqueta de P14).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p15.regimes import (ASSETS, MACRO_COLS, REGIMES, align_panel, balance_vs_rates, correlation_difference,
                                  delayed_publication_demo, diversification_curve, macro_indicators, portfolio_risk,
                                  regime_correlations, regime_summary, synthetic_inputs, systematic_decomposition,
                                  two_asset_variance, validate_macro, validate_returns, zero_variance_weight)

DATA_MODE = 'synthetic'  # synthetic | csv
RETURNS_CSV = 'quantlab_exports/p14_returns_live.csv'
MACRO_CSV = 'macro_point_in_time.csv'
SEED = 42
SPLIT = .65
MAX_AGE_DAYS = 75
EXPORT = False
WEIGHTS = pd.Series([.25,.35,.30,.10], index=ASSETS)
plt.rcParams.update({'figure.figsize':(10,4),'figure.dpi':110,'axes.grid':True,'grid.alpha':.2,'axes.spines.top':False,'axes.spines.right':False})

## 3. Datos y disponibilidad
Esquema de retornos: exportación `p14_returns_live.csv` de P14, columnas `date`, `BTC`, `Acciones`, `Bonos`, `Oro` (retornos semanales en EUR; pueden existir columnas adicionales).

Esquema macro: una fila por actualización del estado conocido, columnas `available_at` (timestamp ISO UTC), `policy_rate_pct`, `inflation_yoy_pct`, `yield_2y_pct`, `yield_10y_pct`, `balance_usd_bn`. Cada fila debe contener valores que ya se conocían conjuntamente en esa hora. El estado anterior sirve para calcular el cambio de tipos y balance. No uses datos revisados retrospectivamente como si se conocieran entonces.

Fuentes públicas sugeridas para construir ese CSV: [ALFRED](https://alfred.stlouisfed.org/) para vintages; [FRED](https://fred.stlouisfed.org/) para series FEDFUNDS, CPIAUCSL, DGS2, DGS10 y WALCL. CPIAUCSL requiere transformar el índice a inflación interanual con versiones disponibles en cada fecha; WALCL (millones USD) debe dividirse entre 1.000. FEDFUNDS es una media mensual efectiva, no el tipo objetivo de una reunión; si se quiere estudiar decisiones de política, utilizar el rango objetivo publicado y sus horas. El calendario de publicación debe verificarse en la fuente primaria. Un vintage con resolución diaria no prueba una hora intradía exacta. No se descarga automáticamente un histórico revisado para simular disponibilidad.

La serie artificial contiene ciclos de política y cambios de correlación introducidos deliberadamente.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿los retornos y el estado macro son utilizables?</b>

- `validate_returns` exige al menos 80 retornos semanales, todos en viernes consecutivos y únicos, finitos y > −100 %.
- `validate_macro` exige al menos 2 actualizaciones, `available_at` en UTC sin duplicados y valores finitos.

Ninguna de las dos rellena huecos: ante un problema, se detienen con un error.

</div>

In [ ]:
print(inspect.getsource(synthetic_inputs))

if DATA_MODE=='synthetic':
    returns,macro=synthetic_inputs(SEED)
elif DATA_MODE=='csv':
    returns=pd.read_csv(RETURNS_CSV,index_col='date')[ASSETS]
    returns.index=pd.to_datetime(returns.index,utc=True,errors='raise').normalize()
    macro=pd.read_csv(MACRO_CSV)
else:
    raise ValueError('DATA_MODE debe ser synthetic o csv')
returns = validate_returns(returns)
macro = validate_macro(macro)
print('Fuente:',DATA_MODE,'| Semanas:',len(returns),'| Actualizaciones macro:',len(macro))
display(returns.head(3),macro.head(3))

## 4. Indicadores y regímenes
$$r_{real,aprox}=i-\pi_{interanual},\quad slope_{bps}=100(y_{10}-y_2)$$
El tipo real ex post aproximado usa inflación pasada; no es la expectativa de inflación futura. La variación de balance es un **proxy** de expansión/contracción, no prueba una política QE/QT (otros factores también modifican el balance). Definimos régimen por cambio del tipo respecto al estado anterior: subida, bajada o sin cambio (tolerancia 0,01 pp). Este régimen elemental no pretende representar el ciclo económico entero.

In [ ]:
print(inspect.getsource(macro_indicators))

macro = macro_indicators(macro)
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
macro.set_index('available_at')[['policy_rate_pct','inflation_yoy_pct','real_policy_proxy_pct']].plot(ax=axes[0]); axes[0].set_ylabel('%'); axes[0].set_title('Tipos e inflación conocidos · '+DATA_MODE)
macro.set_index('available_at').yield_curve_bps.plot(ax=axes[1],color='#376987'); axes[1].axhline(0,color='grey',lw=1); axes[1].set_ylabel('bps'); axes[1].set_title('Pendiente 10 años − 2 años'); fig.tight_layout(); plt.show()

### 4.1. Unión temporal sin datos futuros
El retorno etiquetado viernes t en P14 cubre desde el último cierre común de la semana anterior hasta el último cierre común de la semana actual. Como P14 exporta etiquetas semanales, no la hora exacta del cierre, usamos un corte conservador: lunes 00:00 UTC de la **semana anterior** al viernes t. Así no utilizamos publicaciones de después de ese cierre, ni siquiera en semanas festivas ordinarias. Para un backtest de ejecución real necesitaremos timestamps exactos y precios ejecutables. Las filas sin estado conocido o con más de 75 días de antigüedad se excluyen del análisis por régimen (no se rellenan con cero).

In [ ]:
print(inspect.getsource(align_panel))

panel, cutoff = align_panel(returns, macro, WEIGHTS, MAX_AGE_DAYS, SPLIT)
print('Corte de validación:',cutoff.date(),'| Sin régimen conocido:',(~panel.known_regime).sum())
display(panel.groupby(['split','regime'],dropna=False).size().rename('semanas'))

## 5. Diversificación: riesgo y rendimiento
Para pesos fijos y rebalanceo al inicio de cada semana, $r_p=w^Tr$. La varianza semanal es $w^T\Sigma w$. La contribución a la volatilidad del activo i es $w_i(\Sigma w)_i/\sigma_p$. Puede ser negativa si reduce el riesgo conjunto. Anualizamos volatilidad con $\sqrt{52}$ como convención aproximada que supone ausencia de dependencia temporal relevante; los retornos medios anualizados son aritméticos. No modelamos costes de rebalanceo, impuestos ni viabilidad operativa. Pesos elegidos a priori, no optimizados.

In [ ]:
print(inspect.getsource(portfolio_risk))

period_vol,contribution=portfolio_risk(returns,WEIGHTS)
portfolio=returns@WEIGHTS
risk=pd.DataFrame({'peso_pct':WEIGHTS*100,'volatilidad_anual_pct':returns.std()*np.sqrt(52)*100,
                   'contribucion_vol_anual_pp':contribution*np.sqrt(52)*100,'contribucion_pct_del_riesgo':contribution/period_vol*100})
display(risk.round(2))
print('Volatilidad anual cartera:',round(period_vol*np.sqrt(52)*100,2),'%')
print('Volatilidad individual ponderada:',round((returns.std()@WEIGHTS)*np.sqrt(52)*100,2),'%')
equity=(1+returns.assign(Cartera=portfolio)).cumprod()*100
baseline=pd.DataFrame([np.full(len(equity.columns),100.)],columns=equity.columns,index=[equity.index[0]-pd.Timedelta(days=7)])
equity=pd.concat([baseline,equity])
fig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True)
equity.plot(ax=axes[0]); axes[0].set_title('Capital base 100 EUR · '+DATA_MODE); axes[0].set_ylabel('EUR por 100 iniciales')
(equity/equity.cummax()-1).mul(100)[['BTC','Acciones','Cartera']].plot(ax=axes[1]); axes[1].set_title('Drawdown desde máximo'); axes[1].set_ylabel('%'); fig.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La diversificación reduce el riesgo, pero BTC lo domina</b>

| Activo | Peso | Volatilidad anual | Contribución al riesgo | % del riesgo total |
|---|---|---|---|---|
| BTC | 25 % | 42,1 % | 9,69 pp | **64,3 %** |
| Acciones | 35 % | 17,0 % | 4,38 pp | 29,0 % |
| Bonos | 30 % | 11,1 % | 0,98 pp | 6,5 % |
| Oro | 10 % | 6,6 % | 0,03 pp | 0,2 % |

- La cartera tiene una volatilidad del **15,1 %**, frente al **20,5 %** de la media ponderada de las volatilidades individuales. Esos 5,4 puntos son el efecto de la diversificación: los activos no se mueven perfectamente juntos.
- **BTC tiene un 25 % del peso, pero aporta casi dos tercios del riesgo.** Un peso no es lo mismo que una contribución al riesgo.
- El oro apenas aporta riesgo: su correlación con el resto es casi nula en esta simulación.

</div>

### 5.1. Riesgo sistemático e idiosincrático
Regresión ilustrativa: $r_{BTC,t}=\alpha+\beta r_{acciones,t}+\epsilon_t$. Usamos el retorno de acciones en EUR como proxy de un factor, no de todo el mercado ni un CAPM de retornos excedentes. Ajustamos en desarrollo y calculamos errores en validación. En la muestra de ajuste, la ortogonalidad de OLS permite descomponer la varianza observada entre componente explicado y residual. El residual puede contener factores omitidos, por lo que no es necesariamente riesgo idiosincrático puro.

In [ ]:
print(inspect.getsource(systematic_decomposition))

dev=panel[panel.split=='Desarrollo']; val=panel[panel.split=='Validación']
sysdec = systematic_decomposition(dev, val, 'BTC', 'Acciones')
alpha, beta, x, resid = sysdec['alpha'], sysdec['beta'], sysdec['x'], sysdec['resid']
systematic, residual, total = sysdec['systematic_var'], sysdec['residual_var'], sysdec['total_var']
print(f'Beta de BTC frente a acciones: {beta:.3f} | Varianza explicada: {systematic/total:.1%}')
print(f"Error de validación RMSE: {sysdec['val_rmse']:.2%}")
y = dev.BTC.to_numpy()
fig,ax=plt.subplots(figsize=(7,4.5)); ax.scatter(x*100,y*100,s=13,alpha=.5,label='Desarrollo'); grid=np.linspace(x.min(),x.max(),80); ax.plot(grid*100,(alpha+beta*grid)*100,color='#bb6c38',label='Ajuste OLS'); ax.set(xlabel='Retorno semanal acciones (%)',ylabel='Retorno semanal BTC (%)',title='Exposición de BTC al factor acciones · '+DATA_MODE); ax.legend(); fig.tight_layout(); plt.show()

## 6. ¿Cambia el comportamiento por régimen?
Comparamos retornos medios semanales y volatilidad por estado conocido. No inferimos causalidad: el entorno puede coincidir con otros factores. Usamos el mismo conjunto de reglas en validación y marcamos grupos con menos de 12 observaciones como insuficientes para esta comparación descriptiva.

In [ ]:
print(inspect.getsource(regime_summary))

cols=ASSETS+['portfolio_return']
known=panel[panel.known_regime]
summary=known.groupby(['split','regime'])[cols].agg(['count','mean','std'])
means, counts = regime_summary(panel, cols, 12)
for split in ['Desarrollo','Validación']:
    print(split,'| semanas por régimen:',counts.reindex(pd.MultiIndex.from_product([[split],REGIMES])).fillna(0).astype(int).tolist())
    display(means.reindex(pd.MultiIndex.from_product([[split],REGIMES])).round(3))
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,split in zip(axes,['Desarrollo','Validación']):
    block=means.reindex(pd.MultiIndex.from_product([[split],REGIMES])).droplevel(0)
    block[['BTC','Acciones','Bonos','portfolio_return']].rename(columns={'portfolio_return':'Cartera'}).plot.bar(ax=ax,rot=0)
    ax.set_title(split); ax.set_ylabel('Retorno semanal medio (%)'); ax.axhline(0,color='grey',lw=.7); ax.set_xlabel('Régimen conocido')
    for i,rr in enumerate(REGIMES):
        nobs=counts.get((split,rr),0)
        if nobs<12: ax.text(i,.03,'n insuf.',transform=ax.get_xaxis_transform(),ha='center',fontsize=8)
fig.suptitle('Retornos condicionales · '+DATA_MODE); fig.tight_layout(); plt.show()

### 6.1. Correlaciones en diferentes entornos
Una cartera diversificada en condiciones normales puede perder parte de su protección si las correlaciones aumentan durante periodos de tensión. No se identifica una crisis solo por una subida de tipos. Medimos la correlación entre acciones y bonos por régimen, y la evolución móvil de sus correlaciones.

In [ ]:
corr_table = regime_correlations(panel, 'Acciones', 'Bonos', 12)
display(corr_table.round(3))
rolling=returns.Acciones.rolling(26,min_periods=26).corr(returns.Bonos)
fig,ax=plt.subplots(); rolling.plot(ax=ax,color='#376987'); ax.axhline(0,color='grey',lw=1); ax.axvline(cutoff,color='#bb6c38',ls='--',label='Inicio validación'); ax.set_ylim(-1,1); ax.set_ylabel('Correlación Pearson'); ax.set_title('Acciones y bonos · 26 semanas móviles · '+DATA_MODE); ax.legend(); fig.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Durante las subidas, los bonos dejan de proteger</b>

Retorno semanal medio de la cartera y correlación acciones–bonos por régimen:

| Periodo | Régimen | Semanas | Cartera | Correlación acciones–bonos |
|---|---|---|---|---|
| Desarrollo | Subida | 74 | −0,055 % | **+0,55** |
| Desarrollo | Estable | 74 | +0,174 % | −0,38 |
| Desarrollo | Bajada | 52 | +0,013 % | −0,25 |
| Validación | Subida | 27 | −0,414 % | **+0,80** |
| Validación | Estable | 31 | −0,326 % | −0,44 |
| Validación | Bajada | 52 | +0,086 % | −0,52 |

- **En las subidas de tipos, la correlación acciones–bonos se vuelve positiva** (+0,55 y +0,80), y en el resto es negativa. Justo cuando las acciones caen, los bonos también: la diversificación falla cuando más se necesita. Es el efecto que programamos en la simulación.
- La cartera rinde peor en las subidas en los dos periodos. Pero en validación el régimen "Estable" también pierde (−0,33 % semanal), y los retornos medios por régimen no son estables. Con 27–74 semanas por grupo, las medias tienen mucho ruido.
- No se infiere causalidad: el régimen de tipos puede coincidir con otros factores.

</div>

## 7. Comprobaciones automáticas
Los tests no certifican la calidad económica del CSV ni la validez causal; solo comprueban propiedades de las fórmulas y la alineación temporal.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: fórmulas de cartera, OLS, unión sin datos futuros y separación cronológica
checks={
 'pesos suman uno':np.isclose(WEIGHTS.sum(),1),
 'retorno ponderado':np.allclose(panel.portfolio_return,returns.to_numpy()@WEIGHTS.to_numpy()),
 'varianza cartera':np.isclose(period_vol,portfolio.std()),
 'contribuciones suman volatilidad':np.isclose(contribution.sum(),period_vol),
 'correlación perfecta':np.isclose(np.sqrt(.5**2*.4**2+.5**2*.15**2+2*.5*.5*.4*.15),.275),
 'covarianza cero':np.isclose(np.sqrt(.5**2*.4**2+.5**2*.15**2),np.sqrt(.045625)),
 'OLS descompone varianza':np.isclose(systematic+residual,total),
 'OLS residual ortogonal':abs(np.dot(resid,x))<1e-9,
 'sin datos futuros':(panel.loc[panel.available_at.notna(),'available_at']<=panel.loc[panel.available_at.notna(),'decision_at']).all(),
 'staleness acotada':((panel.decision_at-panel.available_at).dropna()<=pd.Timedelta(days=MAX_AGE_DAYS)).all(),
 'separación cronológica':dev.date.max()<val.date.min(),
 'capital inicial explícito':(equity.iloc[0]==100).all(),
}
for name,passed in checks.items(): assert passed,name
print(f'{len(checks)} comprobaciones superadas')
if EXPORT:
    out=Path('quantlab_exports'); out.mkdir(exist_ok=True)
    panel.assign(data_mode=DATA_MODE).to_csv(out/f'p15_panel_{DATA_MODE}.csv',index=False)
    summary.to_csv(out/f'p15_regime_summary_{DATA_MODE}.csv')

## 8. Ejercicios
**E1.** Implementa la varianza de dos activos sin usar `cov`. Prueba correlaciones -1, 0 y +1; explica cuándo la varianza puede ser cero.

**E2.** Calcula la aportación al riesgo de cada activo con otros pesos (40/30/20/10). ¿Cuál tiene mayor contribución? No selecciones pesos mirando validación.

**E3.** Simula N activos con un factor común y ruido independiente. Representa el riesgo de una cartera equiponderada conforme N aumenta. ¿Qué parte no desaparece?

**E4.** Implementa la pendiente 10y–2y y el tipo real aproximado. Investiga cómo clasificar una curva invertida y por qué no garantiza recesión.

**E5.** Compara la contracción y expansión del balance con subidas y bajadas de tipos. Explica por qué el balance no identifica QE/QT por sí solo.

**E6.** Modifica el umbral de antigüedad a 30 días. Registra cuántas filas se pierden; compara una unión incorrecta por fecha de observación con la unión por disponibilidad usando una publicación artificial retrasada 15 días.

**E7.** Importa los retornos reales de P14 y construye el CSV macro con vintages y fechas de publicación verificadas. Guarda fuentes, unidades, versión y método de alineación. Si faltan datos, registra el bloqueo en lugar de inventarlos.

**E8.** Formula antes de mirar la validación una hipótesis sobre correlación de acciones y bonos durante subidas de tipos. Reporta tamaño de muestra, diferencia observada, incertidumbre, factores omitidos y limitaciones de costes. No ajustes umbrales en el periodo reservado.

### E1 · Varianza de dos activos

In [ ]:
print(inspect.getsource(two_asset_variance))

assert np.isclose(two_asset_variance(.5,.4,.15,1),.275**2)
assert np.isclose(two_asset_variance(.5,.4,.15,0),.045625)
assert np.isclose(two_asset_variance(.5,.2,.2,-1),0)
display(pd.DataFrame({rho: {'varianza': two_asset_variance(.5, .4, .15, rho), 'volatilidad': np.sqrt(two_asset_variance(.5, .4, .15, rho))}
                      for rho in (-1, 0, 1)}).T.rename_axis('correlación').round(6))
w0 = zero_variance_weight(.4, .15)
print(f"Con ρ = −1, la varianza es 0 si el peso en A es σ_b / (σ_a + σ_b) = {w0:.4f} → varianza {two_asset_variance(w0, .4, .15, -1):.2e}")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Solo con correlación −1 se puede eliminar el riesgo</b>

50 % en A (σ = 40 %) y 50 % en B (σ = 15 %):

| Correlación | Varianza | Volatilidad |
|---|---|---|
| −1 | 0,0156 | **12,5 %** |
| 0 | 0,0456 | 21,4 % |
| +1 | 0,0756 | **27,5 %** (la media ponderada) |

- Con **ρ = +1** no hay diversificación: la volatilidad es la media ponderada (27,5 %).
- Con **ρ = 0** la volatilidad cae al 21,4 %, y con **ρ = −1**, al 12,5 %.
- **La varianza solo puede ser cero si ρ = −1** y los pesos compensan exactamente las volatilidades: w_A = σ_B / (σ_A + σ_B) = **27,3 %** en A. Con volatilidades iguales (20 % y 20 %), basta un 50/50. En mercados reales no existen dos activos con correlación exactamente −1 de forma estable.

</div>

### E2 · Contribuciones con pesos 40/30/20/10

In [ ]:
w_e2 = pd.Series([.40, .30, .20, .10], index=ASSETS)  # fijados a priori, sin mirar la validación
vol_e2, contribution_e2 = portfolio_risk(returns, w_e2)
display(pd.DataFrame({'peso_pct': w_e2 * 100, 'contribucion_pp': contribution_e2 * np.sqrt(52) * 100,
                      'pct_del_riesgo': contribution_e2 / vol_e2 * 100}).round(2))
print(f"Volatilidad de la cartera: {vol_e2 * np.sqrt(52) * 100:.2f} % (con 25/35/30/10: {period_vol * np.sqrt(52) * 100:.2f} %)")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Subir BTC al 40 % le da más del 80 % del riesgo</b>

| Activo | Peso | Contribución | % del riesgo (40/30/20/10) | % del riesgo (25/35/30/10) |
|---|---|---|---|---|
| BTC | 40 % | 16,35 pp | **80,9 %** | 64,3 % |
| Acciones | 30 % | 3,38 pp | 16,7 % | 29,0 % |
| Bonos | 20 % | 0,45 pp | 2,2 % | 6,5 % |
| Oro | 10 % | 0,03 pp | 0,1 % | 0,2 % |

- **BTC tiene la mayor contribución**, y crece más que su peso: pasar del 25 % al 40 % de peso lleva su parte del riesgo del 64 % al **81 %**. La volatilidad de la cartera sube del 15,1 % al **20,2 %**.
- La contribución depende de la volatilidad del activo y de su covarianza con el resto, no solo del peso. Los pesos se fijaron de antemano: elegirlos mirando la validación la convertiría en parte del desarrollo.

</div>

### E3 · El riesgo que no desaparece al diversificar

In [ ]:
print(inspect.getsource(diversification_curve))

e3 = diversification_curve((1, 2, 5, 10, 20, 50, 100, 200))
display(e3.round(4))
fig, ax = plt.subplots()
ax.plot(e3.index, e3.vol_simulated * 100, marker='o', label='Simulada')
ax.plot(e3.index, e3.vol_theory * 100, ls='--', label='Teoría: √(β²σ²_f + σ²_ε / N)')
ax.axhline(e3.systematic_floor.iloc[0] * 100, color='firebrick', ls=':', label='Suelo sistemático (β·σ_f)')
ax.set(xscale='log', xlabel='Número de activos (N)', ylabel='Volatilidad semanal (%)', title='Diversificación con un factor común'); ax.legend(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Lo que queda es el riesgo del factor común</b>

Cada activo = factor común (β = 1, σ_f = 2 % semanal) + ruido propio (σ_ε = 4 %):

| N | 1 | 2 | 5 | 10 | 20 | 50 | 100 | 200 |
|---|---|---|---|---|---|---|---|---|
| Volatilidad simulada | 4,46 % | 3,50 % | 2,65 % | 2,37 % | 2,16 % | 2,06 % | 2,02 % | 1,99 % |

- σ²_p = β²σ²_f + σ²_ε / N. El **ruido propio** (riesgo idiosincrático) se reduce con 1/N y casi desaparece con 50 activos.
- **El riesgo sistemático no desaparece:** por muchos activos que se añadan, la volatilidad converge al **2 %** del factor común. Diversificar entre activos que dependen del mismo factor (por ejemplo, muchas criptomonedas que siguen a BTC) no elimina ese riesgo.
- La mayor parte de la reducción ocurre con los primeros 10–20 activos.

</div>

### E4 · Pendiente de la curva y tipo real

In [ ]:
inverted = macro[macro.yield_curve_bps < 0]
print(f"Meses con curva invertida: {len(inverted)} de {len(macro)} | pendiente media por régimen (bps):",
      macro.groupby('regime').yield_curve_bps.mean().round(1).to_dict())
print(f"Tipo real aproximado: de {macro.real_policy_proxy_pct.min():.2f} a {macro.real_policy_proxy_pct.max():.2f} pp")
display(macro[['available_at', 'policy_rate_pct', 'inflation_yoy_pct', 'real_policy_proxy_pct', 'yield_curve_bps', 'regime']].head(8)
        .round({'policy_rate_pct': 2, 'inflation_yoy_pct': 2, 'real_policy_proxy_pct': 2, 'yield_curve_bps': 1}))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · La curva invertida es una señal, no una predicción</b>

- **Pendiente** = 100 · (y₁₀ − y₂) bps; **tipo real aproximado** = tipo oficial − inflación interanual pasada (ex post).
- En la simulación, la curva está invertida (**−40 bps**) en **26 de 74 meses**: todos los de las fases de subida. En las pausas, +20 bps; en las bajadas, +80 bps. El tipo real aproximado va de −0,3 a +2,8 pp.
- **Cómo clasificar una inversión:** pendiente < 0 en un par de plazos fijado de antemano (10 − 2 años o 10 años − 3 meses), con datos del momento. Conviene exigir que dure (por ejemplo, varias semanas) para no reaccionar a un solo día.
- **Por qué no garantiza recesión:** refleja que el mercado espera tipos más bajos en el futuro, a menudo porque anticipa una desaceleración, pero también puede deberse a primas de plazo bajas, compras del banco central o demanda de bonos largos. El retraso hasta una posible recesión es largo y variable, y ha habido inversiones sin recesión posterior. Con pocos ciclos históricos, no hay muestra para una regla fiable.

</div>

### E5 · Balance frente a tipos

In [ ]:
print(inspect.getsource(balance_vs_rates))

display(balance_vs_rates(macro))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · En la simulación coinciden siempre; en la realidad, no</b>

| Régimen de tipos | Contracción del balance | Estable | Expansión |
|---|---|---|---|
| Subida | **25** | 0 | 0 |
| Estable | 0 | **24** | 0 |
| Bajada | 0 | 0 | **24** |

- En la simulación, el balance se contrae siempre que suben los tipos y se expande siempre que bajan, porque lo programamos así.
- **Por qué el balance no identifica QE/QT por sí solo:** el balance del banco central también cambia por razones que no son de política monetaria: préstamos de emergencia a bancos, operaciones con otros bancos centrales, variaciones de la cuenta del Tesoro o vencimientos de activos que no se reinvierten. Una subida del balance puede ser un rescate de liquidez y no un QE. Para identificar QE/QT hacen falta los **anuncios** de compra o reducción de activos y su calendario, no solo el saldo.

</div>

### E6 · Antigüedad máxima y publicación retrasada

In [ ]:
print(inspect.getsource(delayed_publication_demo))

for age in (MAX_AGE_DAYS, 30):
    p_age, _ = align_panel(returns, macro, WEIGHTS, age, SPLIT)
    print(f"Antigüedad máxima {age} días → semanas sin régimen conocido: {(~p_age.known_regime).sum()}")
demo = delayed_publication_demo(12, 15)
display(demo)
print("Semanas en que la unión por fecha de referencia usaría un dato aún no publicado:",
      int((demo.known_by_reference & ~demo.known_by_availability).sum()))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Unir por la fecha de referencia mete el futuro en el pasado</b>

- **Umbral de 30 días:** las semanas sin régimen conocido pasan de **3 a 8**. En la simulación, los datos se publican una vez al mes, así que entre una publicación y la siguiente el estado puede tener hasta 30,5 días de antigüedad. Un umbral de 30 días descarta 5 semanas que en realidad tenían un estado válido.
- **Publicación artificial retrasada:** el dato del mes de **marzo de 2024** se publica el **15 de abril**. Unido por la fecha de referencia, se "conoce" desde el 1 de marzo, durante **7 semanas** (del 1 de marzo al 12 de abril) en que todavía no existía. Unido por disponibilidad, solo aparece desde la semana del 19 de abril.
- Esas 7 semanas de *look-ahead* bastan para que un backtest "anticipe" movimientos usando un dato que nadie tenía.

</div>

### E7 · Retornos reales de P14 y datos macro point-in-time
Cambia `LIVE_E7` a `True` para descargar los retornos reales de P14 (Yahoo Finance, requiere `yfinance`) y analizar la parte de la cartera que no depende de datos macro. Si la descarga falla, la celda se detiene con el error.

In [ ]:
LIVE_E7 = False

if LIVE_E7:
    from quantlab.p14.multi_asset import download_prices, prepare_weekly
    _, live_eur, _ = prepare_weekly(download_prices('2021-01-01', '2026-01-01'), '2021-01-01', '2026-01-01')
    live_returns = live_eur.pct_change(fill_method=None).iloc[1:][ASSETS]
    live_returns.index = live_returns.index.tz_localize('UTC')
    live_returns = validate_returns(live_returns)
    print(f"Retornos reales de P14: {len(live_returns)} semanas | {live_returns.index.min().date()} a {live_returns.index.max().date()} | EUR")
    live_vol, live_contribution = portfolio_risk(live_returns, WEIGHTS)
    display(pd.DataFrame({'peso_pct': WEIGHTS * 100, 'volatilidad_anual_pct': live_returns.std() * np.sqrt(52) * 100,
                          'pct_del_riesgo': live_contribution / live_vol * 100}).round(2))
    print(f"Volatilidad de la cartera: {live_vol*np.sqrt(52)*100:.2f} % | media ponderada: {(live_returns.std()@WEIGHTS)*np.sqrt(52)*100:.2f} %")
    live_cut = live_returns.index[int(len(live_returns) * SPLIT)]
    live_dev, live_val = live_returns[live_returns.index < live_cut], live_returns[live_returns.index >= live_cut]
    live_sys = systematic_decomposition(live_dev, live_val, 'BTC', 'Acciones')
    print(f"BTC frente a acciones (desarrollo hasta {live_cut.date()}): beta {live_sys['beta']:.3f} | "
          f"varianza explicada {live_sys['explained_share']:.1%} | RMSE de validación {live_sys['val_rmse']:.2%}")
    live_equity = (1 + live_returns @ WEIGHTS).cumprod()
    live_mdd = (live_equity / live_equity.cummax().clip(lower=1) - 1).min()
    live_roll = live_returns.Acciones.rolling(26, min_periods=26).corr(live_returns.Bonos)
    print(f"Cartera: {live_equity.iloc[-1] - 1:+.2%} | caída máxima {live_mdd:.2%} | correlación acciones–bonos: "
          f"desarrollo {live_dev.Acciones.corr(live_dev.Bonos):.3f}, validación {live_val.Acciones.corr(live_val.Bonos):.3f}, "
          f"móvil de 26 semanas de {live_roll.min():.2f} a {live_roll.max():.2f}")
else:
    print("LIVE_E7 = False: no se descarga nada. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Retornos reales sí; datos macro point-in-time, bloqueados</b>

**Retornos reales de P14** (Yahoo Finance, cierres ajustados convertidos a EUR, descarga del 19-11-2026): 258 semanas, del 22-01-2021 al 26-12-2025.

| Activo | Peso | Volatilidad anual | % del riesgo de la cartera |
|---|---|---|---|
| BTC | 25 % | 57,4 % | **76,7 %** |
| Acciones (SPY) | 35 % | 15,8 % | 18,1 % |
| Bonos (IEF) | 30 % | 9,1 % | 3,6 % |
| Oro (GLD) | 10 % | 14,0 % | 1,6 % |

- Volatilidad de la cartera: **17,2 %**, frente a un 24,0 % de media ponderada. La cartera ganó un 103,8 % con una caída máxima del 28,7 %.
- **BTC frente a acciones** (desarrollo hasta el 05-04-2024): beta **0,98**, pero las acciones explican solo el **5,5 %** de la varianza de BTC. Casi todo su riesgo es propio, o depende de factores que este modelo no incluye.
- En el conjunto de cada periodo, la correlación acciones–bonos fue **positiva** (+0,15 en desarrollo, +0,18 en validación), aunque la móvil de 26 semanas osciló entre −0,16 y +0,68. Es coherente con un periodo dominado por las subidas de tipos de 2022–2023, pero no se puede atribuir sin el análisis por régimen.

**Bloqueo: el CSV macro point-in-time no se ha construido.** Lo que falta y por qué:

| Dato necesario | Fuente | Bloqueo |
|---|---|---|
| Vintages de FEDFUNDS, CPIAUCSL, DGS2, DGS10 y WALCL | ALFRED (API de FRED) | La API requiere una clave personal que no está configurada; la descarga CSV de FRED da la serie **revisada de hoy**, no lo que se sabía en cada fecha. |
| Hora de publicación de cada dato | Calendarios oficiales (BLS para el IPC, Fed para el balance H.4.1, Tesoro para los rendimientos) | Hay que verificarla en la fuente primaria, dato a dato. |
| Decisiones de tipos (no la media mensual FEDFUNDS) | Comunicados del FOMC con su fecha y hora | No hay una serie descargable con las horas exactas. |

Usar la serie revisada con un retraso supuesto sería **inventar** la disponibilidad. El análisis por régimen con datos reales queda pendiente hasta tener la clave de ALFRED y el calendario verificado. Método de alineación previsto: el mismo `align_panel` (estado disponible antes del lunes de la semana anterior, antigüedad máxima de 75 días).

</div>

### E8 · Hipótesis sobre la correlación acciones–bonos durante las subidas

In [ ]:
print(inspect.getsource(correlation_difference))

# Hipótesis registrada ANTES de mirar la validación (ver la conclusión de abajo).
display(pd.DataFrame([correlation_difference(panel, 'Subida', 'Acciones', 'Bonos', period) for period in ('Desarrollo', 'Validación')]).round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Hipótesis, contraste y límites</b>

**Hipótesis (fijada antes de mirar la validación):** *durante las semanas en régimen de subida de tipos, la correlación entre los retornos semanales de acciones y bonos es mayor que en el resto de semanas.* Umbral de régimen: el de la sección 4 (±0,01 pp), sin cambios. Contraste: diferencia de correlaciones con la transformada de Fisher y un IC del 95 %.

| Periodo | Semanas en subida | Resto | Correlación en subida | Resto | Diferencia (r) | Diferencia (z) | IC 95 % (z) |
|---|---|---|---|---|---|---|---|
| Desarrollo | 74 | 126 | +0,55 | −0,32 | +0,87 | **+0,95** | [0,66; 1,25] |
| Validación | 27 | 83 | **+0,80** | −0,49 | +1,28 | **+1,62** | [1,16; 2,07] |

- **La hipótesis se mantiene en validación:** la diferencia en escala z es positiva y su intervalo no incluye el 0. En la simulación es esperable, porque el efecto está **programado** (los bonos dependen positivamente del mercado durante las subidas).
- **Incertidumbre:** el intervalo de Fisher supone semanas independientes. Con dependencia temporal y solo 27 semanas de subida en validación, es optimista.
- **Factores omitidos:** inflación, nivel de tipos, primas de plazo, liquidez y shocks de crecimiento pueden cambiar la correlación por sí mismos y coincidir con las subidas.
- **Costes:** una estrategia que redujera bonos durante las subidas pagaría rebalanceos, spreads e impuestos, que no están modelados. Además, el régimen se conoce con retraso (el lunes de la semana anterior).
- **Con datos reales**, la correlación de 2021–2025 fue positiva en general (≈ +0,15), pero sin régimen point-in-time (E7) la hipótesis **no se puede contrastar todavía**.

</div>

## 9. Conclusiones y siguiente paso

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P15</b>

**¿Cuánto riesgo reduce la cartera?** En la simulación, la volatilidad baja del 20,5 % (media ponderada) al 15,1 %; con datos reales, del 24,0 % al 17,2 %. Es lo que aportan las correlaciones imperfectas.

**¿Qué riesgo permanece?** El sistemático, el de los factores comunes (E3). Y la concentración: BTC, con un 25 % del peso, aporta el 64 % del riesgo en la simulación y el 77 % con datos reales.

**¿Se mantienen los resultados entre periodos?** La relación entre régimen y correlación sí (programada en la simulación). Los retornos medios por régimen, no del todo: el régimen «Estable» pasa de +0,17 % a −0,33 % semanal entre desarrollo y validación.

**¿Qué ocurre si cambian las correlaciones?** La diversificación se debilita justo cuando más falta: en las subidas de tipos, acciones y bonos pasan a moverse juntos (correlación +0,55 a +0,80), y la cartera pierde su colchón.

**¿Qué datos reales necesitamos para comprobarlo?** Un histórico macro point-in-time (vintages de ALFRED, con su clave de API), las horas de publicación verificadas en las fuentes primarias, las decisiones de tipos del FOMC con su hora, y los retornos de P14 con timestamps exactos de cierre. Sin eso, el análisis por régimen con datos reales queda bloqueado (E7).

Los regímenes, pesos y efectos sintéticos son didácticos; no hemos estimado una ventaja de inversión.

📄 Resumen en `docs/conclusiones/P15_macro_regimes.md`

</div>

En P16 ampliaremos la teoría de derivados y la construcción de factores.